In [1]:
!git clone https://github.com/ihy-adi/DS-GAN-MajorProject.git

Cloning into 'DS-GAN-MajorProject'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 38 (delta 7), reused 12 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 32.02 MiB | 13.45 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [2]:
%cd DS-GAN-MajorProject

/content/DS-GAN-MajorProject


In [3]:
!pip install thop torchsummary


In [9]:
!ls


 Files	'Final Model'   gan.py	'Old model'   README.md


In [13]:
import torch
from thop import profile
from generator_only import Generator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Generator().to(device)
model.load_state_dict(
    torch.load("Final Model/generator_final.pth", map_location=device),
    strict=False
)

model.eval()

# Params
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Params: {params:.2f} M")

# FLOPs
dummy = torch.randn(1, 3, 256, 256).to(device)
flops, _ = profile(model, inputs=(dummy,), verbose=False)
print(f"FLOPs: {flops/1e9:.2f} G")

# FPS
import time
for _ in range(10): model(dummy)
torch.cuda.synchronize()

start = time.time()
for _ in range(50): model(dummy)
torch.cuda.synchronize()

fps = 50 / (time.time() - start)
print(f"FPS: {fps:.2f}")


Params: 1.60 M
FLOPs: 15.74 G
FPS: 80.30
